<a href="https://colab.research.google.com/github/anindyaa25/Project-KKA-NabilaAnindya/blob/main/Kelompok_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np

In [13]:
# DATA LOADING DAN DATA INSPECTION

df = pd.read_csv('dataset_penjualan_kantin.csv')

print(df.head()) #5 baris pertama
print(df.info()) # tipe data dan jumlah non-null tiap kolom
print(df.describe()) # statistik ringkas kolom numerik
print(df.shape) # jumlah (baris, kolom)

  id_transaksi     tanggal  nama_produk kategori jumlah_terjual harga_satuan  \
0      TRX0042  2026-08-11   Roti Bakar  Makanan              9         7000   
1      TRX0005  2026-08-03     Gorengan  makanan              2         2000   
2      TRX0011  2026-08-04  Jus Alpukat  Minuman              4          NaN   
3      TRX0035  2026-08-10     Mie Ayam  Makanan            NaN        10000   
4      TRX0007  2026-08-03      Kerupuk    Snack             10      Rp2.000   

  nama_kasir metode_pembayaran  
0   Pak Agus             Tunai  
1     Bu Sri              QRIS  
2    Bu Wati             Tunai  
3   Pak Joko          Transfer  
4    Bu Wati          Transfer  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaksi       69 non-null     object
 1   tanggal            69 non-null     object
 2   nama_produk        69 non-

In [14]:
# DATA CLEANING
print(df.isnull().sum())  # jumlah data kosong tiap kolom

df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0)  # isi kekosongan dengan 0
df['nama_kasir'] = df['nama_kasir'].fillna('Tidak diketahui')  # isi kekosongan nama kasir
df = df.dropna(subset=['harga_satuan'])  # hapus baris jika kolom harga_satuan kosong

print(df.duplicated().sum())  # jumlah baris duplikat
df = df.drop_duplicates()

# --- TAMBAHKAN BAGIAN INI ---
df['harga_satuan'] = (
    df['harga_satuan']
    .astype(str)
    .str.replace('Rp', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# -----------------------------

df['harga_satuan'] = df['harga_satuan'].astype(int)  # memastikan tipe data harga adalah integer
print(df.dtypes)

id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64
4
id_transaksi         object
tanggal              object
nama_produk          object
kategori             object
jumlah_terjual       object
harga_satuan          int64
nama_kasir           object
metode_pembayaran    object
dtype: object


In [16]:
# DATA MANIPULATION

df['jumlah_terjual'] = (
    df['jumlah_terjual']
    .astype(str)
    .str.replace(r'[^0-9]', '', regex=True)
)
df['jumlah_terjual'] = pd.to_numeric(df['jumlah_terjual'], errors='coerce').fillna(0).astype(int)
df_filter = df[df['jumlah_terjual'] > 0]
print(df_filter.head())

df['total_pendapatan'] = df['harga_satuan'] * df['jumlah_terjual']
print(df[['harga_satuan', 'jumlah_terjual', 'total_pendapatan']].head())

df_sorted = df.sort_values(by='total_pendapatan', ascending=False)
print(df_sorted.head(10))

hasil_group = (
    df.groupby('nama_kasir')
    .agg(
        total_terjual=('jumlah_terjual', 'sum'),
        total_pendapatan=('total_pendapatan', 'sum')
    )
    .reset_index()
    .sort_values('total_pendapatan', ascending=False)
)
print(hasil_group)

  id_transaksi          tanggal nama_produk kategori  jumlah_terjual  \
0      TRX0042       2026-08-11  Roti Bakar  Makanan               9   
1      TRX0005       2026-08-03    Gorengan  makanan               2   
4      TRX0007       2026-08-03     Kerupuk    Snack              10   
5      TRX0045       2026-08-11   Teh Botol  Minuman               9   
6      TRX0061  14 Agustus 2026   Teh Botol  MINUMAN              14   

   harga_satuan nama_kasir metode_pembayaran  
0          7000   Pak Agus             Tunai  
1          2000     Bu Sri              QRIS  
4          2000    Bu Wati          Transfer  
5          5000   Pak Joko              QRIS  
6          5000     Bu Sri              QRIS  
   harga_satuan  jumlah_terjual  total_pendapatan
0          7000               9             63000
1          2000               2              4000
3         10000               0                 0
4          2000              10             20000
5          5000               9    

In [17]:
# DATA PROFILING SUMMARY

raw = pd.read_csv('dataset_penjualan_kantin.csv')

baris_awal = len(raw)
baris_akhir = len(df)
kosong_jumlah = raw['jumlah_terjual'].isnull().sum()
kosong_harga = raw['harga_satuan'].isnull().sum()
kosong_kasir = raw['nama_kasir'].isnull().sum()
duplikat = raw.duplicated().sum()

total_omzet = df['total_pendapatan'].sum()
rata_transaksi = df['total_pendapatan'].mean()

kasir = df.groupby('nama_kasir')['total_pendapatan'].sum().sort_values(ascending=False)
kasir = kasir.drop('Tidak diketahui', errors='ignore')
kasir_top = kasir.index[0]
persen_top = kasir.iloc[0] / total_omzet * 100

def rp(x):
    return 'Rp' + f"{x:,.0f}".replace(',', '.')

print("Temuan 1 (kualitas data):")
print(f"Dataset awal berisi {baris_awal} baris. Ditemukan {kosong_jumlah} data kosong pada jumlah_terjual, "
      f"{kosong_harga} pada harga_satuan, {kosong_kasir} pada nama_kasir, dan {duplikat} baris duplikat. "
      f"Setelah cleaning, tersisa {baris_akhir} baris yang siap dianalisis.")

print("\nTemuan 2 (pendapatan):")
print(f"Total pendapatan kantin pada data ini sebesar {rp(total_omzet)}, "
      f"dengan rata-rata {rp(rata_transaksi)} per transaksi.")

print("\nTemuan 3 (kasir):")
print(f"Kasir {kasir_top} mencatat pendapatan tertinggi, yaitu sekitar {persen_top:.1f}% dari total pendapatan.")

Temuan 1 (kualitas data):
Dataset awal berisi 69 baris. Ditemukan 4 data kosong pada jumlah_terjual, 3 pada harga_satuan, 3 pada nama_kasir, dan 4 baris duplikat. Setelah cleaning, tersisa 62 baris yang siap dianalisis.

Temuan 2 (pendapatan):
Total pendapatan kantin pada data ini sebesar Rp6.596.000, dengan rata-rata Rp106.387 per transaksi.

Temuan 3 (kasir):
Kasir Pak Joko mencatat pendapatan tertinggi, yaitu sekitar 69.4% dari total pendapatan.


In [18]:

df.to_csv('dataset_bersih.csv', index=False)

from google.colab import files
files.download('dataset_bersih.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

6. Tahap cleaning paling menantang, karena tipe data beberapa kolom tidak sesuai. Kolom harga_satuan berisi teks berformat "Rp" dan titik, dan jumlah_terjual terbaca sebagai teks sehingga tidak bisa dihitung. Kami mengatasinya dengan menghapus simbol lalu mengubah tipe datanya jadi angka. Ini contoh dari yang tadi kamu alami, jadi ganti kalau kelompokmu merasa tahap lain yang paling sulit.

Setiap cara menangani data kosong punya dampak berbeda. Menghapus baris mengurangi jumlah data, sedangkan mengisi nilai bisa mengubah makna data. Di dataset ini, jumlah_terjual kosong diisi 0 karena dianggap tidak terjual, nama_kasir kosong diisi "Tidak diketahui" supaya barisnya tetap terpakai, dan baris dengan harga_satuan kosong dihapus karena harganya tidak bisa ditebak dan akan merusak perhitungan pendapatan. Kalau dilakukan asal-asalan, hasil analisis bisa menyesatkan

Data analyst di dunia nyata sebagian besar waktunya dipakai untuk membersihkan dan menyiapkan data sebelum dianalisis. Dataset bersih dari proyek ini adalah bahan dasar untuk visualisasi dan keputusan bisnis, karena kualitas insight bergantung pada kualitas datanya.